In [10]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

import secret

import pathlib
import textwrap
from IPython.display import display
from IPython.display import Markdown

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = secret.GEMINI_API_KEY

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))


In [15]:
stock_id = "2912"

folder_path = "stock_news/"+stock_id+"/"+stock_id+"_news/"

files = os.listdir(folder_path)
sorted_files = sorted(files)
sorted_files

['20030624',
 '20030810',
 '20030909',
 '20040629',
 '20041117',
 '20041118',
 '20060321',
 '20060326',
 '20060331',
 '20060726',
 '20070801',
 '20071031',
 '20071204',
 '20080330',
 '20080930',
 '20090113',
 '20090607',
 '20090629',
 '20100214',
 '20100415',
 '20100418',
 '20100628',
 '20100719',
 '20111102',
 '20120416',
 '20120722',
 '20120828',
 '20121021',
 '20130104',
 '20130117',
 '20130118',
 '20130222',
 '20130516',
 '20130528',
 '20130617',
 '20130705',
 '20130711',
 '20130716',
 '20130906',
 '20131024',
 '20131029',
 '20131113',
 '20131218',
 '20131219',
 '20131226',
 '20140106',
 '20140208',
 '20140211',
 '20140223',
 '20140224',
 '20140309',
 '20140312',
 '20140314',
 '20140401',
 '20140414',
 '20140508',
 '20140513',
 '20140515',
 '20140518',
 '20140802',
 '20140908',
 '20140912',
 '20140915',
 '20140916',
 '20140917',
 '20140918',
 '20140919',
 '20140924',
 '20140926',
 '20140928',
 '20140929',
 '20141003',
 '20141008',
 '20141009',
 '20141011',
 '20141013',
 '20141014',

In [43]:
import pandas as pd
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory
from langchain_core.messages import HumanMessage, SystemMessage
from datetime import datetime

signal = []

p_mutated = '強勁的財務表現、媒體廣泛讚譽其的創新和光明前景，預示著該公司的股價將持續上漲。'
general_prompt = '請依據' + p_mutated + '給這篇文章打一個分數，分數介於-100～100之間，-100是負向，100是最好，0則是無法判斷'
llm = ChatGoogleGenerativeAI(
        model="gemini-pro", 
        temperature=1,
        safety_settings={
            HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
        })

for index, file_name in enumerate(sorted_files):
    scores = []
    
    # 檢查日期是否在參考日期之後
    try:
        date_obj = datetime.strptime(file_name, '%Y%m%d')
    except:
        continue
    if date_obj < datetime(2022, 1, 1):
        continue
    elif date_obj > datetime(2022, 2, 15):
        continue
    
    with open(folder_path + file_name) as f:
        news =  f.read()
        prompt = general_prompt + "\n" + news
    
    for i in range(1):
        result = llm.invoke(prompt)
        try:
            scores.append(int(result.content))
        except:
            pass
    
    if len(scores) == 0:
        score = 0
    else:
        score = sum(scores)/len(scores)
    
    
    signal.append([date_obj.strftime('%Y%m%d'),score])
    
    

In [44]:
#加入買入價格
df = pd.read_csv("price_history/"+stock_id+".csv", dtype=str)
price_np = df.to_numpy()
prices = price_np.tolist()
for item in signal:
    
    for price in prices:
        # print(item[0], price[0])
        if item[0] == price[0]:
            print(item)

# print([file_name,score])
    
    # w_folder_path = "stock_news/"+stock_id+"/"+stock_id+"_response/"
    # w_file_path = stock_id+"r_" + sorted_files[i].split('_')[1]
    # os.makedirs(w_folder_path, exist_ok=True)
    # 
    # with open(w_folder_path+w_file_path, "w") as f:
    #     f.write(result.content)

['20220104', 0.0]
['20220105', 0.0]
['20220110', 0.0]
['20220111', 0.0]
['20220112', 0.0]
['20220118', 0.0]
['20220119', 0.0]
['20220124', 0.0]
['20220125', 0.0]
['20220126', 0]
['20220211', 100.0]
['20220214', 0.0]
['20220215', 0.0]


In [20]:
#test cell
with open("prompt template/temp3", "r") as f:
    prompt_temp = f.read()

p1 = "，並打一個分數，分數介於0～100之間，100代表越正向，0分是負向，50分是中立"
p2 = "請用一句話產生一個選股條件"
llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=1)

f = open("stock_news/test1", "r")
news =  f.read()
prompt = prompt_temp + "\n" + news

result = llm.invoke(p2)

print(f'prompt: {prompt}')
print("__")
print(f'response: {result.content}')
print("_"*50)

prompt: 請給這篇文章打一個分數，分數介於0～100之間越高代表情緒越好

輝達 (NVDA-US) 在 GTC 大會發表最新一代 AI 晶片 Blackwell GPU，其中由 2 片 B200 與 1 顆 Grace CPU 組成的 GB200，預計將在今年下半年量產。法人指出，伺服器機殼廠勤誠 (8210-TW) 與鴻海 (2317-TW) 將合作設計，成為 GB200 主要的機殼設計廠，有望推升勤誠營運。
法人表示，勤誠將生產長版與短版 2 種機殼規格，單價也將較傳統 2U 伺服器高。
法人進一步指出，過去勤誠伺服器客戶主要是亞馬遜 (AMZN-US)AWS，但成為 GB200 的主要供應商後，包括鴻海、台系與美系等伺服器組裝廠都成為勤誠客戶，且未來勤誠成為 GB200 主要供應商後，客戶將以美系四大 CSP 業者為主。
至於一般伺服器方面，勤誠於法說會時指出，雖然一般伺服器庫存去化告一段落，但由於受到 AI 伺服器的排擠效應，目前訂單能見度不高，復甦力道仍疲弱，下半年才有望回升。不過法人預期，勤誠在 GB200、AI/ASIC 等產品的挹注下，營運可維持成長。
根據調查，B200 熱功耗高達 1000 至 2000 瓦，對水冷散熱的依賴度將提高，預計 B200 水冷散熱的滲透率可能達 50%。法人認為，GB200 也將大量採用水冷散熱。勤誠執行長陳亞男於法說會時指出，今年散熱技術專案仍以氣冷為主，不過已有水冷型以及低噪音的產品正在開發當中。另外，也積極與 ODM 合作開發水冷與液冷產品，預計 1 至 2 年待相關技術成熟後才會出貨。
__
response: `WHERE 年份 >= 2000`
__________________________________________________


In [48]:
p1 = "請判斷公司管理層具有長期思維和明顯的產業優勢，"
p2 = "公司被新聞正面報導，且財務指標持續成長，"

p = "請將提示1" + p1 + "提示2" + p2 + "交叉上述提示並生成一個新的提示;突變步驟1中生成的提示符，並生成以<prompt>和</prompt>為括號的最終提示符"
for i in range(2):
    llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=1)
    result = llm.invoke(p)
    print(result.content)

<prompt>公司管理層具有長期思維和明顯的產業優勢，被新聞正面報導，且財務指標持續成長，因而展現出強勁的投資潛力。</prompt>
<prompt>基於提示 1 中偵測到的長期思維和明確產業優勢，以及提示 2 中提及的正面新聞報導和持續的財務成長，我假設該公司具有</prompt>堅實的競爭優勢和良好的經營前景</prompt>。


In [ ]:
p1 = "公司管理層具有長期思維和明顯的產業優勢，被新聞正面報導，且財務指標持續成長，因而展現出強勁的投資潛力。"
p2 = "堅實的競爭優勢和良好的經營前景"

p = "請將提示1" + p1 + "提示2" + p2 + "交叉上述提示並生成一個新的提示;突變步驟1中生成的提示符，並生成以<prompt>和</prompt>為括號的最終提示符"
for i in range(2):
    llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=1)
    result = llm.invoke(p)
    print(result.content)

In [50]:
p1 = "請判斷公司管理層具有長期思維和明顯的產業優勢，"
p2 = "公司被新聞正面報導，且財務指標持續成長，"

p = "請將提示1" + p1 + "提示2" + p2 + "交叉上述提示並生成一個新的提示;突變步驟1中生成的提示符，並生成以<prompt>和</prompt>為括號的最終提示符，重複以上的步驟10次"
for i in range(2):
    llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=1)
    result = llm.invoke(p)
    print(result.content)

**步驟 1**
提示 1: 公司管理層具有長期思維和明顯的產業優勢
提示 2: 公司被新聞正面報導，且財務指標持續成長

**步驟 2**
交叉提示 1 和提示 2:
具有長期思維和明顯產業優勢的管理層的公司，被正面新聞報導並持續財務增長的。

**步驟 3**
突變步驟 1 中生成的提示符:
<prompt>提示 1: 公司管理層具有長期思維和明顯的產業優勢 <br>提示 2: 公司被新聞正面報導，且財務指標持續成長</prompt>

**步驟 4**
生成新的提示符:
<prompt>具有長期思維和持續產業優勢的管理層，且最近被新聞正面報導並持續財務增長的公司。</prompt>

**重複步驟 1-4，以下依序為步驟 5-10 的提示符:**

**步驟 5**
<prompt>具有長期思維和持續產業優勢的管理層，且最近被新聞正面報導並持續財務增長的，未來發展前景看好且穩定成長的公司。</prompt>

**步驟 6**
<prompt>具有長期思維和持續產業優勢的管理層，且最近被新聞正面報導並持續財務增長的，市佔率持續擴大且擁有獨特的競爭優勢的公司。</prompt>

**步驟 7**
<prompt>具有長期思維和持續產業優勢的管理層，且最近被新聞正面報導並持續財務增長的，具有創新能力和持續研發的公司。</prompt>

**步驟 8**
<prompt>具有長期思維和持續產業優勢的管理層，且最近被新聞正面報導並持續財務增長的，擁有強健的品牌形象和忠誠顧客的公司。</prompt>

**步驟 9**
<prompt>具有長期思維和持續產業優勢的管理層，且最近被新聞正面報導並持續財務增長的，在全球市場具有競爭力和持續擴張的公司。</prompt>

**步驟 10**
<prompt>具有長期思維和持續產業優勢的管理層，且最近被新聞正面報導並持續財務增長的，具有社會責任和永續經營理念的公司。</prompt>
**提示 1：** 公司管理層具有長期思維和明顯的產業優勢。

**提示 2：** 公司被新聞正面報導，且財務指標持續成長。

**交叉提示：** 公司被正面報導且財務狀況穩健，管理層具有長遠願景和產業優勢，表明其具有良好的投資潛力。

**新提示：**

1. **<prompt>提示 3：** 公司在競爭激烈的市場中保持領先地位，並持續

In [51]:
p1 = "請判斷公司管理層具有長期思維和明顯的產業優勢，"
p2 = "公司被新聞正面報導，且財務指標持續成長，"

p = "請將提示1" + p1 + "提示2" + p2 + "交叉上述提示並生成一個新的提示;突變步驟1中生成的提示符，並生成以<prompt>和</prompt>為括號的最終提示符，重複以上的步驟10次"
for i in range(2):
    llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=1)
    result = llm.invoke(p)
    print(result.content)

**步驟 1**

提示 1：公司管理層具有長期思維和明顯的產業優勢。

提示 2：公司被新聞正面報導，且財務指標持續成長。

**交叉後提示：**

公司管理層具有長期思維、產業優勢顯著，且公司受到媒體積極關注，財務業績持續改善。

**<prompt>步驟 1 提示：公司管理層具有長期思維和明顯的產業優勢；公司被新聞正面報導，且財務指標持續成長。交叉後提示：公司管理層具有長期思維、產業優勢顯著，且公司受到媒體積極關注，財務業績持續改善。</prompt>**

**步驟 2**

提示 1：交叉後提示：公司管理層具有長期思維、產業優勢顯著，且公司受到媒體積極關注，財務業績持續改善。

提示 2：公司獲利能力強，具有高毛利率。

**交叉後提示：**

公司管理層具有長期思維、產業優勢顯著，且公司受到媒體積極關注、財務業績持續改善，並具有高毛利率，獲利能力強。

**<prompt>步驟 2 提示：公司管理層具有長期思維和明顯的產業優勢；公司被新聞正面報導，且財務指標持續成長。交叉後提示：公司管理層具有長期思維、產業優勢顯著，且公司受到媒體積極關注，財務業績持續改善。交叉後提示：公司管理層具有長期思維、產業優勢顯著，且公司受到媒體積極關注、財務業績持續改善，並具有高毛利率，獲利能力強。</prompt>**

（重複此過程 8 次）

**步驟 10**

提示 1：交叉後提示：公司管理層具有長期思維、產業優勢顯著，且公司受到媒體積極關注、財務業績持續改善、獲利能力強、產品具有競爭優勢。

提示 2：公司積極進行股東收益，包括股票回購和股息發放。

**交叉後提示：**

公司管理層具有長期思維、產業優勢顯著，且公司受到媒體積極關注、財務業績持續改善、獲利能力強、產品具有競爭優勢，並積極進行股東收益，包括股票回購和股息發放。

**<prompt>步驟 1 提示：公司管理層具有長期思維和明顯的產業優勢；公司被新聞正面報導，且財務指標持續成長。交叉後提示：公司管理層具有長期思維、產業優勢顯著，且公司受到媒體積極關注，財務業績持續改善。交叉後提示：公司管理層具有長期思維、產業優勢顯著，且公司受到媒體積極關注、財務業績持續改善，並具有高毛利率，獲利能力強。交叉後提示：公司管理層具有長期思維、產業優勢顯著，且公司受到媒體積極關注、財務業績持續改善，並具有高毛利率，獲利能力

In [52]:
p1 = "請判斷公司管理層具有長期思維和明顯的產業優勢，"
p2 = "公司被新聞正面報導，且財務指標持續成長，"

p = "請將提示1" + p1 + "提示2" + p2 + "交叉上述提示並生成一個新的提示;突變步驟1中生成的提示符，並生成以<prompt>和</prompt>為括號的最終提示符，重複以上的步驟10次"
for i in range(2):
    llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=1)
    result = llm.invoke(p)
    print(result.content)

**步驟 1**
**提示 1：** 公司管理層具有長期思維和明顯的產業優勢。
**提示 2：** 公司被新聞正面報導，且財務指標持續成長。

**交叉後的提示：** 公司管理層具有長期思維、產業優勢明顯，且公司財務表現持續佳、新聞評價正面。

**<prompt>**公司管理層具有長期思維、產業優勢明顯，且公司財務表現持續佳、新聞評價正面。**</prompt>

**步驟 2**
**提示 1：** 公司擁有專利技術，市場競爭力強。
**提示 2：** 公司產品銷量持續成長，獲利能力優異。

**交叉後的提示：** 公司擁有專利技術、競爭力強，且產品銷量及獲利能力持續優異。

**<prompt>公司擁有專利技術、競爭力強，且產品銷量及獲利能力持續優異，同時管理層具有長期思維、產業優勢明顯，且公司財務表現持續佳、新聞評價正面。**</prompt>

**步驟 3**
**提示 1：** 公司經營團隊經驗豐富，營運效率佳。
**提示 2：** 公司內部控制完善，財報可信度高。

**交叉後的提示：** 公司經營團隊經驗豐富、營運效率佳，且內部控制完善、財報可信度高。

**<prompt>公司經營團隊經驗豐富、營運效率佳，且內部控制完善、財報可信度高，同時擁有專利技術、競爭力強，且產品銷量及獲利能力持續優異，管理層具有長期思維、產業優勢明顯，且公司財務表現持續佳、新聞評價正面。**</prompt>

以此類推，重複上述步驟 10 次，即可逐漸完善提示符。
**步驟 1**

* 提示 1：公司管理層具有長期思維和明顯的產業優勢
* 提示 2：公司被新聞正面報導，且財務指標持續成長

* 交叉提示：公司管理層具有長期思維、產業優勢明顯，且被新聞正面報導，財務表現良好

**步驟 2**

* 提示 3：公司積極投資研發

* 交叉提示：公司管理層具有長期思維、產業優勢明顯，被新聞正面報導，財務表現良好，且積極投資研發

**步驟 3**

* 提示 4：公司持續推出創新產品

* 交叉提示：公司管理層具有長期思維、產業優勢明顯，被新聞正面報導，財務表現良好，積極投資研發，並持續推出創新產品

**步驟 4**

* 提示 5：公司擁有忠誠的客戶群

* 交叉提示：公司管理層具有長期思維、產業優勢明顯，被新聞正面報導，財務表現良好，積極投資研發，持續推出創新產品，

In [10]:
prompt= '產生以下指令的1個變體，同時保留語意：'
p1 = "由於公司強勁且穩定的財務表現，以及媒體對其創新和長期前景的廣泛贊譽，預計股價將穩步提升。"

llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=1)
result = llm.invoke(prompt+p1)
print(result.content)

強勁的財務表現、媒體廣泛讚譽其的創新和光明前景，預示著該公司的股價將持續上漲。


In [9]:
#test cell
p = '強勁的財務表現、媒體廣泛讚譽其的創新和光明前景，預示著該公司的股價將持續上漲。'
general_prompt = '請依據' + p + '給這篇文章打一個分數，分數介於0～100之間，0是負向，100是最好，50則是無法判斷'


llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=1)

f = open("stock_news/test1", "r")
news =  f.read()
prompt = general_prompt + "\n" + news

for i in range(100):
    result = llm.invoke(prompt)
    
    # print(f'prompt: {prompt}')
    # print("__")
    print(f'response: {result.content}')
    print("_"*50)

response: 50
__________________________________________________
response: 50
__________________________________________________
response: 50
__________________________________________________
response: 50
__________________________________________________
response: 50
__________________________________________________
response: 50
__________________________________________________
response: 50
__________________________________________________
response: 50
__________________________________________________
response: 50
__________________________________________________
response: 0
__________________________________________________
response: 無法判斷，50
__________________________________________________
response: 50
__________________________________________________
response: 50
__________________________________________________
response: 50
__________________________________________________
response: 50
__________________________________________________
response: 無法判斷：50
__________________

KeyboardInterrupt: 